# Man-in-the-Middle — Hijack-Pursuit in a Fleet

Three drones fly straight-north missions from spread-out homes, all monitored
by a **single GCS process** (they share one `gcs_name`). An attacker has a
man-in-the-middle interposed on **one** drone's link. Mid-mission the MITM
switches that victim to GUIDED and streams setpoints chasing a *different*
fleet member's live Remote ID position — a hijack-pursuit against a peer.

The attacker learns the target's position passively by overhearing the Remote
ID feed the compromised drone already receives (no Oracle changes). The
target must stay within transmission range of the victim for the chase to
keep tracking.

In [1]:
from simulator import Simulator
from simulator.config import DATA_PATH, PARAMS_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Origin, fleet layout, and waypoints

In [2]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

speed = 5.0        # m/s
cruise_alt = 10.0  # m — one shared altitude layer
model = Model.IRIS

# Three drones on one GCS, packed close together so all are visible at once.
sysids = [1, 2, 3]
colors = [Color.GREEN, Color.BLUE, Color.RED]
homes = ENUPose.list(
    [  # east, north, up, heading
        (-10.0, 0.0, 0.0, 0),
        (0.0, 0.0, 0.0, 0),
        (10.0, 0.0, 0.0, 0),
    ]
)

# Each drone flies the same short north mission relative to its own home.
# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_20,
#              seq=3 flying to north_40.
home_wp  = ENU(x=0, y=0,  z=0)
north_20 = ENU(x=0, y=20, z=cruise_alt)
north_40 = ENU(x=0, y=40, z=cruise_alt)
mission_wps = [home_wp, north_20, north_40]

# The attacker compromises the red drone (sysid 3) and steers it to pursue
# the green drone (sysid 1). Trigger right after takeoff so the chase starts
# while the target is still close by.
victim_sysid = 3
target_sysid = 1
trigger_seq = 2

## Vehicles (all on one shared GCS)

In [3]:
# A single shared gcs_name puts all three vehicles under one GCS process.
gcs_name = f"FLEET_{Color.BLUE.emoji}"

mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

vehs: list[SimVehicle] = []
for sysid, color, home in zip(sysids, colors, homes, strict=True):
    mission_path = str(mission_folder / f"mitm_pursuit_fleet_{sysid}.waypoints")
    plan = AutoPlan.from_relative_path(
        name="north_mission",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=home,
        relative_path=mission_wps,
        mission_path=mission_path,
        navigation_speed=speed,
        firmware=model.firmware,
    )
    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=gcs_name,
        plan=plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=home,
        relative_path=mission_wps,
        model=model,
    )
    vehs.append(veh)

## Visualizer

In [4]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)

## Simulator + selective MITM hijack-pursuit

In [5]:
simulator = Simulator(visualizer=gaz, verbose=1)

# The victim gets avoidance-disabled params so it truly closes on its target;
# every other drone keeps normal params.
for veh in vehs:
    parm = "mallicious.parm" if veh.sysid == victim_sysid else "vehicle.parm"
    simulator.add_vehicle(veh, parm=str(PARAMS_PATH / parm))

# `simulator.mitm` is keyed by sysid, so the MITM is interposed on ONLY the
# victim. The other drones share the same GCS but talk to it directly.
simulator.mitm[victim_sysid] = {
    "strategy": "hijack_pursuit",
    "params": {
        "target_sysid": target_sysid,
        "trigger_seq": trigger_seq,
        "update_interval": 0.5,
    },
}

simulator.show()

In [6]:
orac = simulator.launch()
orac.run()

14:19:34 - Oracle ⚪ - INFO - 🖥️  Gazebo launched for realistic simulation and 3D visualization.
14:19:34 - Oracle ⚪ - INFO - 🚀 GCS FLEET_🟦 launched (PID 33440)
14:19:34 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 3 vehicles and 1 GCSs
14:19:37 - mitm_3 - INFO - MITM proxy active for vehicle 3 (strategy=HijackPursuitStrategy)
14:19:38 - GCS_FLEET_🟦 - INFO - Vehicle 1 connected
14:19:38 - logic_1 - INFO - Logic 🧠 1: launching
14:19:38 - GCS_FLEET_🟦 - INFO - Vehicle 3 connected
14:19:38 - logic_3 - INFO - Logic 🧠 3: launching
14:19:38 - GCS_FLEET_🟦 - INFO - Vehicle 2 connected
14:19:38 - GCS_FLEET_🟦 - INFO -  GCS FLEET_🟦 started with 3 Vehicles
14:19:38 - GCS_FLEET_🟦 - INFO - Monitoring Vehicle 1
14:19:38 - GCS_FLEET_🟦 - INFO - Monitoring Vehicle 2
14:19:38 - GCS_FLEET_🟦 - INFO - Monitoring Vehicle 3
14:19:38 - logic_2 - INFO - Logic 🧠 2: launching
14:19:38 - logic_3 - INFO - 🧹 Vehicle 3: Cleared previous mission
14:19:38 - logic_1 - INFO - 🧹 Vehicle 1: Cleared previous mission
14:19:38 - 

Connection reset or closed by peer on TCP socket
Connection reset or closed by peer on TCP socket
Connection reset or closed by peer on TCP socket


14:21:02 - logic_3 - INFO - Vehicle 3 logic stopped
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", line 334, in <module>
    main()
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", line 78, in main
    start_logic(config)
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", line 199, in start_logic
    logic.act()
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", line 246, in act
    time.sleep(0.01)  # Avoid busy loop if plan.act() returns immediately
    ^^^^^^^^^^^^^^^^
KeyboardInterrupt
14:21:02 - logic_1 - INFO - Vehicle 1 logic stopped
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", line 334, in <module>
    main()
  File "/home/ubuntu/uav-cyber-sim/simulator/logic.py", li

KeyboardInterrupt: 

## What to observe

- **Green (sysid 1)** and **blue (sysid 2)** fly their full north missions.
- **Red (sysid 3)** starts north, then breaks off and chases the green drone,
  continuously re-targeting green's live Remote ID position — driven entirely
  by the man-in-the-middle, with no command from the shared GCS.
- The chase tracks only while green stays within transmission range of red.